In [1]:
import json
from pathlib import Path
from typing import Any, Dict, List

import cloudpickle

# IMPORTANT: Ensure RL4CRN classes + task kinds are registered BEFORE unpickling
from RL4CRN.utils import default_tasks  # noqa: F401


# ----------------------------
# Config
# ----------------------------
SAVED_TRAINERS_ROOT = Path("saved_trainers")
DOCS_APPS_ROOT = Path("docs") / "Applications"

SORT_HOF_BY_REWARD = True

WRITE_NAV_SNIPPET = True
NAV_SNIPPET_PATH = Path("docs") / "_generated_nav_applications.yml"


# ----------------------------
# Helpers
# ----------------------------
def load_checkpoint(path: Path) -> Dict[str, Any]:
    with open(path, "rb") as f:
        payload = cloudpickle.load(f)
    if not isinstance(payload, dict):
        raise ValueError(f"{path} did not contain a dict-like checkpoint payload.")
    return payload


def get_task_kind(payload: Dict[str, Any]) -> str:
    cfg = payload.get("config", {}) or {}
    task = cfg.get("task", {}) or {}
    kind = task.get("kind", None)
    return str(kind) if kind is not None else "unknown"


def unwrap_crn(obj: Any) -> Any:
    """
    Unwrap common wrappers (e.g., env-like objects) to get the underlying CRN.
    """
    if hasattr(obj, "state"):
        return obj.state
    return obj


def reward_of(crn: Any) -> float:
    info = getattr(crn, "last_task_info", None) or {}
    if "reward" not in info:
        raise KeyError("CRN.last_task_info has no 'reward' key.")
    return float(info["reward"])


def require_to_md(crn: Any) -> str:
    """
    Strict: must have to_md() and it must return non-empty markdown.
    No fallbacks, no caching, no swallowing errors.
    """
    fn = getattr(crn, "to_md", None)
    if not callable(fn):
        raise AttributeError(
            f"{type(crn).__name__} has no callable to_md(). "
            f"Implement to_md() (or ensure you're unpickling the expected class)."
        )
    md = fn()  # let exceptions propagate
    if not isinstance(md, str) or not md.strip():
        raise ValueError(f"{type(crn).__name__}.to_md() returned empty / non-string output.")
    return md.rstrip() + "\n"


def md_path_for_pkl(pkl_path: Path) -> Path:
    rel = pkl_path.relative_to(SAVED_TRAINERS_ROOT)
    return (DOCS_APPS_ROOT / rel).with_suffix(".md")


def find_all_pkls() -> List[Path]:
    return sorted(SAVED_TRAINERS_ROOT.rglob("*.pkl"))


# ----------------------------
# Markdown page generator (STRICT)
# ----------------------------
def build_md_page(payload: Dict[str, Any]) -> str:
    task_kind = get_task_kind(payload)
    hof = payload.get("hall_of_fame_crns", None)
    if hof is None:
        raise KeyError("Checkpoint payload has no 'hall_of_fame_crns' key.")
    crns_raw = list(hof)

    # unwrap early and validate
    crns = [unwrap_crn(x) for x in crns_raw]
    if not crns:
        raise ValueError("Hall of Fame is empty (no CRNs to write).")

    if SORT_HOF_BY_REWARD:
        # strict reward read (raise if missing)
        crns = sorted(crns, key=reward_of)

    lines: List[str] = []
    lines.append(f"# {task_kind}")
    lines.append("")

    for i, crn in enumerate(crns, start=1):
        r = reward_of(crn)  # strict
        lines.append(f"## HoF {i} — reward: `{r:.6g}`")
        lines.append("")
        lines.append(require_to_md(crn))  # strict
        lines.append("")

    return "\n".join(lines)


# ----------------------------
# Optional: mkdocs nav snippet
# ----------------------------
def sanitize_title(s: str) -> str:
    return s.replace("_", " ").strip()


def build_nav_tree(md_files: List[Path]) -> Dict[str, Any]:
    tree: Dict[str, Any] = {}
    for md in md_files:
        rel_to_docs = md.relative_to("docs").as_posix()
        parts = md.relative_to(DOCS_APPS_ROOT).parts
        node = tree
        for part in parts[:-1]:
            node = node.setdefault(part, {})
        node[parts[-1]] = rel_to_docs
    return tree


def tree_to_mkdocs_nav(title: str, tree: Dict[str, Any], indent: int = 0) -> List[str]:
    sp = "  " * indent
    lines: List[str] = []
    lines.append(f"{sp}- {title}:")
    for k in sorted(tree.keys(), key=lambda x: x.lower()):
        v = tree[k]
        if isinstance(v, dict):
            lines.extend(tree_to_mkdocs_nav(sanitize_title(k), v, indent + 1))
        else:
            lines.append(f"{sp}  - {sanitize_title(Path(k).stem)}: {v}")
    return lines


def main():
    pkls = find_all_pkls()
    if not pkls:
        raise FileNotFoundError(f"No .pkl found under {SAVED_TRAINERS_ROOT}")

    md_files: List[Path] = []
    for pkl in pkls:
        payload = load_checkpoint(pkl)

        md_path = md_path_for_pkl(pkl)
        md_path.parent.mkdir(parents=True, exist_ok=True)

        md_text = build_md_page(payload)  # STRICT: any issue raises immediately
        md_path.write_text(md_text, encoding="utf-8")

        md_files.append(md_path)
        print(f"[OK] wrote {md_path}")

    if WRITE_NAV_SNIPPET:
        tree = build_nav_tree(md_files)
        nav_lines = tree_to_mkdocs_nav("Applications", tree, indent=0)
        NAV_SNIPPET_PATH.write_text("\n".join(nav_lines) + "\n", encoding="utf-8")
        print(f"[OK] wrote mkdocs nav snippet -> {NAV_SNIPPET_PATH}")

    print("[DONE]")


if __name__ == "__main__":
    main()

[OK] wrote docs/Applications/DoseResponse/HF3/1_added_reactions/1_added_reactions_run0.md
[OK] wrote docs/Applications/DoseResponse/HF3/2_added_reactions/2_added_reactions_run0.md
[OK] wrote docs/Applications/DoseResponse/HF3/3_added_reactions/3_added_reactions_run0.md
[OK] wrote docs/Applications/DoseResponse/HF3/4_added_reactions/4_added_reactions_run0.md
[OK] wrote docs/Applications/DoseResponse/HF3/5_added_reactions/hill_response_n=3_run0.md
[OK] wrote docs/Applications/DoseResponse/HF3/5_added_reactions/hill_response_n=3_run1.md
[OK] wrote docs/Applications/DoseResponse/HF3/6_added_reactions/6_added_reactions_run0.md
[OK] wrote docs/Applications/DoseResponse/Ultra_sensitivity/Ultra-sensitivity_run0.md
[OK] wrote docs/Applications/DoseResponse/Ultra_sensitivity/Ultra-sensitivity_run1.md
[OK] wrote docs/Applications/DoseResponse/Ultra_sensitivity/Ultra-sensitivity_run2.md
[OK] wrote docs/Applications/Habituation/Habituation_h3_Task_3s_5r.md
[OK] wrote docs/Applications/Habituation/H